# Naive Bayes

**Owner:** Teammate 1

Bernoulli Naive Bayes needs every input to be 0 or 1, so number columns are cut into five bins first (`kind="binned"`).

Rules that apply to every notebook: use `prep` and `evaluate` for the split, the preparation and the scoring. Do not write your own. The test set is only touched inside `evaluate.run_feature_steps`.

In [ ]:
# Course      : CS 513 - Data Mining & Machine Learning
# First Name  : <fill in>
# Last Name   : <fill in>
# Id          : <fill in>
# Purpose     : Naive Bayes on the three feature steps.
#
# AI NOTICE   : The starter code in this notebook was generated by AI (Claude).
#               Prompt and context: docs/AI_PROMPT_LOG.md, entry 1.
#               Mark any cell you add with AI help the same way.

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from src import evaluate, plots, prep

df = prep.load_table()
train, test = prep.split_by_building(df)
print(f"{len(df):,} filings, {df['bin'].nunique():,} buildings, "
      f"{df['is_unsafe'].mean():.1%} Unsafe")
print(f"Train: {len(train):,} rows   Test: {len(test):,} rows")

## Train and test on the three feature steps

In [ ]:
from sklearn.naive_bayes import BernoulliNB

results, fitted = evaluate.run_feature_steps(
    "Naive Bayes", BernoulliNB(),
    {"alpha": [0.1, 0.5, 1.0, 2.0]},
    kind="binned")
results.round(3)

## Which traits raise or lower the chance of Unsafe?

For each 0-1 input, compare how often it is 1 among Unsafe filings against the others. Positive = seen more with Unsafe.

Reading the names: a number column was cut into bins, so `p_age_at_filing_0.0` is the youngest group of buildings and `p_age_at_filing_4.0` the oldest. Rename them for the slide.

In [ ]:
best = fitted["3_plus_violations"].best_estimator_
names = best.named_steps["prep"].get_feature_names_out()
model = best.named_steps["model"]
lift = pd.Series(model.feature_log_prob_[1] - model.feature_log_prob_[0], index=names)
lift = lift.sort_values()
top = pd.concat([lift.head(8), lift.tail(8)])

fig, ax = plots.new_axes("Naive Bayes: traits most tied to an Unsafe rating",
                         xlabel="Log ratio (Unsafe vs not Unsafe)", size=(7.5, 5.5))
colors = [plots.SERIES[1] if v > 0 else plots.SERIES[0] for v in top]
plots.bars(ax, top.index[::-1], top.values[::-1], color=colors)
ax.axvline(0, color=plots.INK_SOFT, linewidth=1)
plt.show()

## To do (Teammate 1)

- Write two or three sentences: do these traits make engineering sense?
- Naive Bayes assumes the inputs are independent. Wall type and material flags are not. Say how that could affect the result.
- Copy your three rows into your slides from `results/comparison_table.csv`.